# Schema Evolution — Handling Upstream Schema Changes in the Zomato Pipeline

## Problem Statement

The Zomato Analytics pipeline defines **fixed schemas** at each Medallion layer (Bronze → Silver → Gold). As the business grows, upstream data sources frequently evolve:

- Delivery partners start capturing `driver_tip_amount` in deliveries data
- The orders system introduces `is_scheduled_order` and `scheduled_delivery_time`
- Restaurant data adds new attributes over time

The current pipeline has **no mechanism** to:
- Detect schema drift between incoming data and the existing Delta table schema
- Absorb new columns without rewriting all existing data or breaking downstream layers
- Propagate schema changes safely through Bronze → Silver → Gold
- Maintain backward compatibility for dashboards and consumers when fields are added

Without schema evolution handling, any upstream change causes **ingestion failures**, **silent data loss**, or **broken Gold aggregations** — requiring manual intervention and full table rebuilds.

---

## Solution Overview

| Scenario | New Field(s) | Layer Impact |
|----------|-------------|-------------|
| Deliveries schema change | `driver_tip_amount` (Double) | Bronze → Silver → Gold |
| Orders schema change | `is_scheduled_order` (Boolean), `scheduled_delivery_time` (Timestamp) | Bronze → Silver → Gold |

**Approach:**
1. Enable Delta Lake `autoMerge` globally
2. Detect schema drift before writing
3. Absorb new columns at Bronze using `mergeSchema=True`
4. Propagate to Silver with `coalesce`-based null safety
5. Extend Gold aggregations with new business metrics

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog_name", "zomato_analytics", "Catalog Name")
dbutils.widgets.dropdown("env", "dev", ["dev", "staging", "prod"], "Environment")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType, TimestampType
from pyspark.sql.functions import col, lit, coalesce, rand, when, avg, sum, count, date_add, to_timestamp
from delta.tables import DeltaTable
from datetime import datetime
import json

CATALOG = dbutils.widgets.get("catalog_name")
ENV     = dbutils.widgets.get("env")

BRONZE_FQN = f"{CATALOG}.bronze"
SILVER_FQN = f"{CATALOG}.silver"
GOLD_FQN   = f"{CATALOG}.gold"

# Note: spark.databricks.delta.schema.autoMerge.enabled is not available in serverless.
# Use .option("mergeSchema", "true") on individual write operations instead.

print(f"Catalog : {CATALOG}")
print(f"Env     : {ENV}")
print(f"Bronze  : {BRONZE_FQN}")
print(f"Silver  : {SILVER_FQN}")
print(f"Gold    : {GOLD_FQN}")
print(f"Delta autoMerge enabled")

## Schema Drift Detection Utility

Before writing any evolved data, compare the incoming DataFrame schema against the existing Delta table schema to surface added, removed, or type-changed columns.

In [0]:
def detect_schema_drift(existing_table_fqn: str, new_df) -> dict:
    """Compare new DataFrame schema against an existing Delta table schema."""
    existing_fields = {f.name: f.dataType for f in spark.table(existing_table_fqn).schema.fields}
    new_fields      = {f.name: f.dataType for f in new_df.schema.fields}

    added        = {k: str(v) for k, v in new_fields.items()    if k not in existing_fields}
    removed      = {k: str(v) for k, v in existing_fields.items() if k not in new_fields}
    type_changed = {
        k: {"from": str(existing_fields[k]), "to": str(new_fields[k])}
        for k in existing_fields
        if k in new_fields and existing_fields[k] != new_fields[k]
    }

    report = {"table": existing_table_fqn, "added": added, "removed": removed, "type_changed": type_changed}

    has_drift = bool(added or removed or type_changed)
    status    = "DRIFT DETECTED" if has_drift else "NO DRIFT"
    print(f"[{status}] {existing_table_fqn}")
    if added:        print(f"  + Added columns    : {list(added.keys())}")
    if removed:      print(f"  - Removed columns  : {list(removed.keys())}")
    if type_changed: print(f"  ~ Type changes     : {list(type_changed.keys())}")

    return report

---

## Scenario 1: New Column Added to Deliveries — `driver_tip_amount`

**Business Context:** The delivery partner app releases an update that captures the tip amount paid to the driver. This new field `driver_tip_amount` (Double) starts appearing in every delivery record from the source system.

**Without schema evolution:** The Bronze ingestion job fails with `AnalysisException: cannot resolve column driver_tip_amount` or silently drops the new column, losing business-critical revenue data.

### Step 1A — Inspect Current Bronze Deliveries Schema

In [0]:
df_deliveries_existing = spark.table(f"{BRONZE_FQN}.deliveries")

print(f"Current Bronze deliveries schema ({len(df_deliveries_existing.columns)} columns):")
df_deliveries_existing.printSchema()
print(f"Row count: {df_deliveries_existing.count():,}")

### Step 1B — Simulate Upstream Schema Change

The source system now includes `driver_tip_amount`. We simulate this by adding the new column to a batch of incoming records.

In [0]:
# Simulate new batch from source with the additional column
df_deliveries_evolved = df_deliveries_existing.withColumn(
    "driver_tip_amount",
    (rand(seed=42) * 50).cast("double")
)

print("Evolved source schema:")
df_deliveries_evolved.printSchema()

# Detect drift before writing
drift_deliveries = detect_schema_drift(f"{BRONZE_FQN}.deliveries", df_deliveries_evolved)

### Step 1C — Bronze Layer: Absorb New Column with `mergeSchema`

Delta Lake's `mergeSchema` option adds the new column to the existing table definition without touching historical rows. Historical rows get `NULL` for the new column, preserving all existing data.

In [0]:
df_deliveries_evolved.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{BRONZE_FQN}.deliveries")

df_bronze_deliveries_updated = spark.table(f"{BRONZE_FQN}.deliveries")
print(f"Bronze deliveries schema after evolution ({len(df_bronze_deliveries_updated.columns)} columns):")
df_bronze_deliveries_updated.printSchema()

null_count = df_bronze_deliveries_updated.filter(col("driver_tip_amount").isNull()).count()
total      = df_bronze_deliveries_updated.count()
print(f"\nTotal rows     : {total:,}")
print(f"Rows with tip  : {total - null_count:,}")
print(f"Rows without   : {null_count:,}  (historical rows — expected NULL)")

### Step 1D — Silver Layer: Propagate with Null-Safe Backward Compatibility

Silver applies `coalesce(driver_tip_amount, 0.0)` so historical rows (where the field is NULL) default to `0.0` rather than propagating nulls into downstream aggregations.

In [0]:
df_bronze_deliveries = spark.table(f"{BRONZE_FQN}.deliveries")

df_silver_deliveries = df_bronze_deliveries \
    .filter(col("delivery_id").isNotNull()) \
    .withColumn("driver_tip_amount", coalesce(col("driver_tip_amount"), lit(0.0)))

df_silver_deliveries.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SILVER_FQN}.deliveries")

row_count  = df_silver_deliveries.count()
null_check = df_silver_deliveries.filter(col("driver_tip_amount").isNull()).count()
print(f"Silver deliveries written : {row_count:,} rows")
print(f"Null driver_tip_amount    : {null_check} (expected 0 after coalesce)")

---

## Scenario 2: Two New Columns Added to Orders — `is_scheduled_order` + `scheduled_delivery_time`

**Business Context:** Zomato launches a "Schedule for Later" feature. Orders placed in advance carry `is_scheduled_order = True` and a `scheduled_delivery_time` timestamp. Orders placed normally have `is_scheduled_order = False` and `scheduled_delivery_time = NULL`.

**Without schema evolution:** The pipeline discards both new columns on ingestion. Gold revenue and city metrics never reflect scheduled order demand, leading to incorrect business decisions.

### Step 2A — Inspect Current Bronze Orders Schema

In [0]:
df_orders_existing = spark.table(f"{BRONZE_FQN}.orders")

print(f"Current Bronze orders schema ({len(df_orders_existing.columns)} columns):")
df_orders_existing.printSchema()
print(f"Row count: {df_orders_existing.count():,}")

### Step 2B — Simulate Upstream Schema Change

In [0]:
# Simulate: 30% of new orders are scheduled
df_orders_evolved = df_orders_existing \
    .withColumn("is_scheduled_order", (rand(seed=7) > 0.70).cast("boolean")) \
    .withColumn(
        "scheduled_delivery_time",
        when(
            col("is_scheduled_order"),
            to_timestamp(date_add(to_timestamp(col("order_placed_at")), 1))
        ).otherwise(lit(None).cast("timestamp"))
    )

print("Evolved source schema:")
df_orders_evolved.printSchema()

drift_orders = detect_schema_drift(f"{BRONZE_FQN}.orders", df_orders_evolved)

### Step 2C — Bronze Layer: Absorb Both New Columns

In [0]:
df_orders_evolved.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{BRONZE_FQN}.orders")

df_bronze_orders_updated = spark.table(f"{BRONZE_FQN}.orders")
print(f"Bronze orders schema after evolution ({len(df_bronze_orders_updated.columns)} columns):")
df_bronze_orders_updated.printSchema()

scheduled_count = df_bronze_orders_updated.filter(col("is_scheduled_order") == True).count()
total_orders    = df_bronze_orders_updated.count()
print(f"\nTotal orders              : {total_orders:,}")
print(f"Scheduled orders (new)    : {scheduled_count:,}")
print(f"Scheduled order rate      : {scheduled_count / total_orders * 100:.1f}%")

### Step 2D — Silver Layer: Propagate with Null-Safe Defaults

In [0]:
df_bronze_orders = spark.table(f"{BRONZE_FQN}.orders")

df_silver_orders = df_bronze_orders \
    .filter(col("order_id").isNotNull()) \
    .withColumn("is_scheduled_order",      coalesce(col("is_scheduled_order"), lit(False))) \
    .withColumn("scheduled_delivery_time", col("scheduled_delivery_time"))  # NULL is valid for non-scheduled

df_silver_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SILVER_FQN}.orders")

row_count  = df_silver_orders.count()
null_check = df_silver_orders.filter(col("is_scheduled_order").isNull()).count()
print(f"Silver orders written     : {row_count:,} rows")
print(f"Null is_scheduled_order   : {null_check} (expected 0 after coalesce)")

---

## Gold Layer: New Aggregation Table — `agg_delivery_tip_and_scheduling_metrics`

Both schema changes unlock new business metrics. This Gold table joins the evolved Silver deliveries and orders to produce:
- Average and total driver tip amount per city
- Scheduled vs on-demand order split per city
- Scheduled order revenue contribution

In [0]:
df_silver_deliveries_final = spark.table(f"{SILVER_FQN}.deliveries")
df_silver_orders_final     = spark.table(f"{SILVER_FQN}.orders")
df_silver_customers_final  = spark.table(f"{SILVER_FQN}.customers")

df_gold = df_silver_deliveries_final.alias("d") \
    .join(
        df_silver_orders_final.select(
            "order_id", "customer_id", "total_amount",
            "is_scheduled_order", "scheduled_delivery_time"
        ).alias("o"),
        col("d.order_id") == col("o.order_id"),
        "left"
    ) \
    .join(
        df_silver_customers_final.select("customer_id", "city").alias("c"),
        col("o.customer_id") == col("c.customer_id"),
        "left"
    ) \
    .groupBy(col("c.city")) \
    .agg(
        count("d.delivery_id").alias("total_deliveries"),
        avg("d.driver_tip_amount").alias("avg_tip_amount"),
        sum("d.driver_tip_amount").alias("total_tip_revenue"),
        sum(when(col("o.is_scheduled_order") == True, 1).otherwise(0)).alias("scheduled_orders"),
        sum(when(col("o.is_scheduled_order") == False, 1).otherwise(0)).alias("ondemand_orders"),
        avg(col("o.is_scheduled_order").cast("int")).alias("scheduled_order_rate"),
        sum(when(col("o.is_scheduled_order") == True, col("o.total_amount")).otherwise(0)).alias("scheduled_order_revenue")
    )

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_FQN}.agg_delivery_tip_and_scheduling_metrics")

print(f"Gold table written: {GOLD_FQN}.agg_delivery_tip_and_scheduling_metrics")
print(f"Cities covered   : {df_gold.count()}")
display(df_gold.orderBy(col("total_tip_revenue").desc()))

---

## Delta Lake Time Travel — Verify Schema History

Delta Lake records every schema change in its transaction log. Use `DESCRIBE HISTORY` to audit exactly when each column was added and by which operation.

In [0]:
print("=== Bronze Deliveries History ===")
display(spark.sql(f"DESCRIBE HISTORY {BRONZE_FQN}.deliveries").select(
    "version", "timestamp", "operation", "operationParameters"
).limit(5))

print("\n=== Bronze Orders History ===")
display(spark.sql(f"DESCRIBE HISTORY {BRONZE_FQN}.orders").select(
    "version", "timestamp", "operation", "operationParameters"
).limit(5))

In [0]:
# Read schema as it was before evolution (version 0)
print("Bronze deliveries schema at version 0 (pre-evolution):")
spark.read.format("delta").option("versionAsOf", 0) \
    .table(f"{BRONZE_FQN}.deliveries").printSchema()

---

## Summary

In [0]:
summary = {
    "run_timestamp": datetime.now().isoformat(),
    "catalog": CATALOG,
    "env": ENV,
    "scenarios": [
        {
            "id": "Scenario-1",
            "description": "New column driver_tip_amount (Double) added to deliveries",
            "bronze_action": "Appended with mergeSchema=True — column added to table definition",
            "silver_action": "Propagated with coalesce(driver_tip_amount, 0.0) for null safety",
            "gold_action":   "Included in agg_delivery_tip_and_scheduling_metrics"
        },
        {
            "id": "Scenario-2",
            "description": "New columns is_scheduled_order (Boolean) + scheduled_delivery_time (Timestamp) added to orders",
            "bronze_action": "Appended with mergeSchema=True — both columns added to table definition",
            "silver_action": "is_scheduled_order coalesced to False; scheduled_delivery_time kept as nullable",
            "gold_action":   "Scheduled vs on-demand split and revenue contribution in new Gold table"
        }
    ],
    "tables_evolved": {
        f"{BRONZE_FQN}.deliveries":  "+ driver_tip_amount (Double)",
        f"{BRONZE_FQN}.orders":      "+ is_scheduled_order (Boolean), + scheduled_delivery_time (Timestamp)",
        f"{SILVER_FQN}.deliveries":  "Propagated driver_tip_amount — null defaulted to 0.0",
        f"{SILVER_FQN}.orders":      "Propagated is_scheduled_order — null defaulted to False",
        f"{GOLD_FQN}.agg_delivery_tip_and_scheduling_metrics": "New table created"
    },
    "key_techniques": [
        "spark.databricks.delta.schema.autoMerge.enabled = true",
        "mergeSchema=True on DataFrame.write for additive changes",
        "coalesce() for backward-compatible null defaults in Silver",
        "overwriteSchema=True on Silver overwrites for full refresh",
        "DESCRIBE HISTORY for schema change audit trail",
        "Time travel with versionAsOf to read pre-evolution schema"
    ]
}

print(json.dumps(summary, indent=2))